In [ ]:
import os
import copy
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import warnings
warnings.filterwarnings("ignore")
import os

In [ ]:
!cp -r "/content/drive/MyDrive/Synthetic_Detection/dataset/frames" /content/

In [ ]:
DATA_DIR   = "/content/frames"
OUTPUT_DIR = "/content/drive/MyDrive/Synthetic_Detection/outputs_rgb"
TEST_DIR   = "/content/drive/MyDrive/Synthetic_Detection/dataset/test"

IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
NUM_WORKERS  = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device   : {DEVICE}")
print(f"Data dir : {DATA_DIR}")
print(f"Outputs  : {OUTPUT_DIR}")

Device   : cuda
Data dir : /content/frames
Outputs  : /content/drive/MyDrive/Synthetic_Detection/outputs_rgb


In [ ]:
def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

In [ ]:
class VideoFrameDataset(Dataset):
    CLASSES = {"ai": 0, "real": 1}
    EXTS    = {".jpg", ".jpeg", ".png", ".bmp"}

    def __init__(self, root_dir, transform=None, fft_mode=False):
        self.root_dir = Path(root_dir)
        self.fft_mode = fft_mode
        self.transform = transform
        self.samples  = []   # (path, label)

        for cls_name, label in self.CLASSES.items():
            cls_dir = self.root_dir / cls_name
            if not cls_dir.exists():
                print(f"[WARN] Not found: {cls_dir}")
                continue
            found = [p for p in sorted(cls_dir.rglob("*"))
                     if p.suffix.lower() in self.EXTS]
            self.samples.extend([(p, label) for p in found])
            print(f"  {cls_name}: {len(found)} frames")

        print(f"Total: {len(self.samples)} | FFT mode: {fft_mode}")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def to_fft(img: Image.Image) -> Image.Image:
        gray = np.array(img.convert("L"), dtype=np.float32)
        mag  = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
        mag  = np.log1p(mag)
        mag  = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return Image.fromarray(np.stack([mag] * 3, axis=-1))

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.fft_mode:
            img = self.to_fft(img)
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
def build_loaders(fft_mode=False):
    # Point directly to the physical splits
    train_dir = os.path.join(DATA_DIR, "train")
    val_dir   = os.path.join(DATA_DIR, "val")

    tr_tf = get_train_transforms() if not fft_mode else get_val_transforms() # if FFT we only use get_val_transforms() because FFT images represent frequency information, not natural image appearance.
    # Operations such as Color Jitter Rotation Random Crop
    # would change the frequency spectrum in unrealistic ways and could destroy the meaningful patterns the FFT is supposed to capture.
    va_tf = get_val_transforms()

    train_ds = VideoFrameDataset(train_dir, transform=tr_tf, fft_mode=fft_mode)
    val_ds   = VideoFrameDataset(val_dir, transform=va_tf, fft_mode=fft_mode)

    tr_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    va_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f"  Train: {len(train_ds)} frames | Val: {len(val_ds)} frames | Batch: {BATCH_SIZE}")
    return tr_loader, va_loader

def build_test_loader(fft_mode=False):
    test_frame_dir = os.path.join(DATA_DIR, "test")

    tf = get_val_transforms()

    test_ds = VideoFrameDataset(
        test_frame_dir,
        transform=tf,
        fft_mode=fft_mode
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    print(f"  Test: {len(test_ds)} frames")

    return test_loader

In [ ]:
def build_resnet18():
    model = models.resnet18(
        weights=models.ResNet18_Weights.IMAGENET1K_V1 # essential meaning of pre-trained
    )

    # Freeze everything
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze last residual block
    for param in model.layer4.parameters():
        param.requires_grad = True

    model.fc = nn.Linear(model.fc.in_features, 2)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable:,} / {total:,}")

    return model.to(DEVICE)

In [ ]:
def train(model, tr_loader, va_loader, tag="rgb"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=7,
        gamma=0.1
    )

    best_f1 = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    patience = 3
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "val_f1": []
    }

    for epoch in range(1, NUM_EPOCHS + 1):
        # ---------------- TRAIN ----------------
        model.train()

        t_loss = 0.0
        t_correct = 0
        t_total = 0

        for batch_idx, (imgs, lbls) in enumerate(tr_loader):
            if batch_idx % 50 == 0:
              print(f"Batch {batch_idx}/{len(tr_loader)}")
            imgs = imgs.to(DEVICE)
            lbls = lbls.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, lbls)

            loss.backward()
            optimizer.step()

            t_loss += loss.item() * imgs.size(0)
            t_correct += (outputs.argmax(1) == lbls).sum().item()
            t_total += imgs.size(0)

        tr_l = t_loss / t_total
        tr_a = t_correct / t_total

        # ---------------- VALIDATION ----------------
        model.eval()

        v_loss = 0.0
        v_correct = 0
        v_total = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, lbls in va_loader:
                imgs = imgs.to(DEVICE)
                lbls = lbls.to(DEVICE)

                outputs = model(imgs)
                loss = criterion(outputs, lbls)

                preds = outputs.argmax(1)

                v_loss += loss.item() * imgs.size(0)
                v_correct += (preds == lbls).sum().item()
                v_total += imgs.size(0)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())

        va_l = v_loss / v_total
        va_a = v_correct / v_total
        va_f1 = f1_score(all_labels, all_preds)

        scheduler.step()

        history["train_loss"].append(tr_l)
        history["val_loss"].append(va_l)
        history["train_acc"].append(tr_a)
        history["val_acc"].append(va_a)
        history["val_f1"].append(va_f1)

        marker = ""

        if va_f1 > best_f1:
            best_f1 = va_f1
            best_wts = copy.deepcopy(model.state_dict())

            patience_counter = 0
            marker = " <- best"

        else:
            patience_counter += 1

        print(
            f"[{tag.upper()}] Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"Train Loss: {tr_l:.4f} | "
            f"Val Loss: {va_l:.4f} | "
            f"Train Acc: {tr_a:.4f} | "
            f"Val Acc: {va_a:.4f} | "
            f"F1: {va_f1:.4f}"
            f"{marker}"
        )

        # Early stopping
        if patience_counter >= patience:
            print(
                f"\nEarly stopping triggered "
                f"after {patience} epochs without improvement."
            )
            break

    ckpt = os.path.join(OUTPUT_DIR, f"resnet18_{tag}_best.pth")
    torch.save(best_wts, ckpt)

    model.load_state_dict(best_wts)

    print(f"\nBest validation F1: {best_f1:.4f}")
    print(f"Saved model to: {ckpt}")

    return model, history

In [ ]:
def evaluate(model, loader, tag=""):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            preds.extend(model(imgs).argmax(1).cpu().numpy())
            truths.extend(lbls.numpy())

    print(f"\n── [{tag}] Frame-level Results ──")
    print(classification_report(truths, preds, target_names=["AI", "Real"]))

    cm = confusion_matrix(truths, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["AI", "Real"], yticklabels=["AI", "Real"], ax=ax)
    ax.set_title(f"Confusion Matrix - {tag}")
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"confusion_{tag}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved -> {path}")

In [ ]:
def evaluate_test_videos(model, test_dir, tag="RGB", fft_mode=False):
    model.eval()
    tf = get_val_transforms()

    # Original video names in test folder usually retain the "videos" suffix
    CLASSES = {"ai_videos": 0, "real_videos": 1}
    label_names = {0: "AI", 1: "Real"}

    NUM_FRAMES = 25 # Fixed 25 frames per video

    video_preds  = []
    video_truths = []

    print(f"\n── [{tag}] Per-Video Test Results (25 Frames Extraction) ──")
    print(f"{'Video':<45} {'True':>6} {'Pred':>6} {'Frames':>8} {'Votes':>20}")
    print("-" * 90)

    for cls_name, true_label in CLASSES.items():
        cls_dir = Path(test_dir) / cls_name
        if not cls_dir.exists():
            print(f"[WARN] Not found: {cls_dir}")
            continue

        video_files = sorted([
            p for p in cls_dir.iterdir()
            if p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}
        ])

        for video_path in video_files:
            cap = cv2.VideoCapture(str(video_path))
            if not cap.isOpened():
                continue

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                cap.release()
                continue

            # Linearly space 30 frames exactly as done in extraction script
            if total_frames <= NUM_FRAMES:
                indices = np.arange(total_frames)
            else:
                indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)

            frame_preds = []

            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if not ret:
                    continue

                # Match training: INTER_AREA resize → BGR to RGB
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

                if fft_mode:
                    img = VideoFrameDataset.to_fft(img)

                tensor = tf(img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    pred = model(tensor).argmax(1).item()
                frame_preds.append(pred)

            cap.release()

            if not frame_preds:
                print(f"[WARN] No frames extracted from {video_path.name}")
                continue

            # Majority vote
            vote_counts = Counter(frame_preds)
            voted_label = vote_counts.most_common(1)[0][0]
            vote_str    = " | ".join(f"{label_names[k]}:{v}" for k, v in sorted(vote_counts.items()))

            video_preds.append(voted_label)
            video_truths.append(true_label)

            correct = "✓" if voted_label == true_label else "✗"
            print(f"{video_path.name:<45} {label_names[true_label]:>6} "
                  f"{label_names[voted_label]:>6} {len(frame_preds):>8}"
                  f"     {vote_str}  {correct}")

    # ── Summary ──
    if not video_truths:
        print("No test videos found or processed.")
        return

    print(f"\n── [{tag}] Video-level Summary ──")
    print(f"Total videos  : {len(video_truths)}")
    print(f"Video accuracy: {accuracy_score(video_truths, video_preds):.4f}")
    print(classification_report(video_truths, video_preds, target_names=["AI", "Real"]))

    # Confusion matrix
    cm = confusion_matrix(video_truths, video_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=["AI", "Real"], yticklabels=["AI", "Real"], ax=ax)
    ax.set_title(f"Video-level Confusion Matrix - {tag}")
    ax.set_ylabel("True"); ax.set_xlabel("Predicted")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"confusion_video_test_{tag}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved → {path}")

In [ ]:
def plot_history(history, tag=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)
    for ax, m in zip(axes, ["loss", "acc"]):
        ax.plot(epochs, history[f"train_{m}"], label="Train")
        ax.plot(epochs, history[f"val_{m}"],   label="Val")
        ax.set_title(f"{m.capitalize()} - {tag}")
        ax.set_xlabel("Epoch"); ax.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"history_{tag}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved -> {path}")

In [ ]:
if __name__ == "__main__":

    # ── Stage 2: RGB ───────────────────────────────────────────
    print("\n" + "=" * 55)
    print("STAGE 2 — ResNet-18 on RGB Frames (FC layer only)")
    print("=" * 55)
    tr_rgb, va_rgb = build_loaders(fft_mode=False)
    model = build_resnet18()
    model, hist_rgb = train(model, tr_rgb, va_rgb, tag="rgb")
    test_rgb = build_test_loader(fft_mode=False)
    evaluate(model, test_rgb, tag="RGB_test")
    plot_history(hist_rgb, tag="RGB")

    # Test on actual test videos
    evaluate_test_videos(model, TEST_DIR, tag="RGB", fft_mode=False)

    # ── Stage 3+4: FFT ─────────────────────────────────────────
    print("\n" + "=" * 55)
    print("STAGE 3+4 — Same Model on FFT Magnitude Frames")
    print("=" * 55)
    tr_fft, va_fft = build_loaders(fft_mode=True)
    rgb_ckpt = os.path.join(OUTPUT_DIR, "resnet18_rgb_best.pth")

    print("\nZero-shot FFT evaluation...")
    model.load_state_dict(torch.load(rgb_ckpt, map_location=DEVICE))
    test_fft = build_test_loader(fft_mode=True)

    evaluate(model, test_fft, tag="FFT_test")

    print("\nDone. Outputs saved to:", OUTPUT_DIR)


STAGE 2 — ResNet-18 on RGB Frames (FC layer only)
  ai: 2096 frames
  real: 2100 frames
Total: 4196 | FFT mode: False
  ai: 448 frames
  real: 450 frames
Total: 898 | FFT mode: False
  Train: 4196 frames | Val: 898 frames | Batch: 32
Trainable params: 8,394,754 / 11,177,538
Batch 0/132
Batch 50/132
Batch 100/132
[RGB] Epoch 01/10 | Train Loss: 0.1300 | Val Loss: 0.4328 | Train Acc: 0.9516 | Val Acc: 0.8530 | F1: 0.8533 <- best
Batch 0/132
Batch 50/132
Batch 100/132
[RGB] Epoch 02/10 | Train Loss: 0.0194 | Val Loss: 0.6128 | Train Acc: 0.9945 | Val Acc: 0.8207 | F1: 0.8034
Batch 0/132
Batch 50/132
Batch 100/132
[RGB] Epoch 03/10 | Train Loss: 0.0132 | Val Loss: 0.5455 | Train Acc: 0.9962 | Val Acc: 0.8430 | F1: 0.8517
Batch 0/132
Batch 50/132
Batch 100/132
[RGB] Epoch 04/10 | Train Loss: 0.0214 | Val Loss: 0.6961 | Train Acc: 0.9926 | Val Acc: 0.7650 | F1: 0.7152

Early stopping triggered after 3 epochs without improvement.

Best validation F1: 0.8533
Saved model to: /content/drive/MyD